In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

In [30]:
# Load data
X_train = pd.read_csv('../DATA/ccle_2019/X_train.csv', index_col=0)
X_test = pd.read_csv('../DATA/ccle_2019/X_test.csv', index_col=0)
y_train_multiclass = pd.read_csv('../DATA/ccle_2019/y_train.csv', index_col=0)
y_test_multiclass = pd.read_csv('../DATA/ccle_2019/y_test.csv', index_col=0)

In [31]:
X_train = X_train.T
X_test = X_test.T

In [32]:
print(f"Training data shape: {X_train.shape}")
print(f"Training labels distribution: {y_train_multiclass.value_counts()}")
print()
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels distribution: {y_test_multiclass.value_counts()}")

Training data shape: (891, 39033)
Training labels distribution: Variant_Type
SNP             498
WT              321
DEL              56
INS              14
DNP               2
Name: count, dtype: int64

Testing data shape: (223, 39033)
Testing labels distribution: Variant_Type
SNP             125
WT               80
DEL              14
INS               4
Name: count, dtype: int64




- WT (Wild Type): no mutation is present, TP53 is in its normal, unaltered form.
- SNP (Single Nucleotide Polymorphism): a single base pair in the DNA sequence has been changed.
- INS (Insertion): one or more nucleotides have been inserted into the DNA sequence.
- DEL (Deletion): one or more nucleotides have been deleted.
- DNP (Double Nucleotide Polymorphism): two adjacent base pairs have been altered.

After loading the train and test data splits, we convert the original multi-class labels into binary labels for the binary classification task (mutated vs. non-mutated). We do this after splitting to preserve the original multi-class labels (y_train_initial and y_test_initial) for the multi-class classification task we do later on.

We define wild-type ('WT') as class 0 (non-mutated) and all other mutation types as class 1 (mutated).


In [47]:
# Check that X and y are aligned with the same samples in the same order
assert all(X_train.index == y_train_multiclass.index), "Sample mismatch between features and labels!"
assert all(X_test.index == y_test_multiclass.index), "Sample mismatch between features and labels!"


y_train = y_train_multiclass.map(lambda x: 0 if x == 'WT' else 1)
y_test = y_test_multiclass.map(lambda x: 0 if x == 'WT' else 1)

print(f"Training data shape: {X_train.shape}")
print(f"Training labels distribution: {y_train.value_counts()}")
print()
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels distribution: {y_test.value_counts()}")

Training data shape: (891, 335)
Training labels distribution: Variant_Type
1               570
0               321
Name: count, dtype: int64

Testing data shape: (223, 335)
Testing labels distribution: Variant_Type
1               143
0                80
Name: count, dtype: int64


# Selection of target genes

In [34]:
tp53_targets = [
    'CDKN1A', 'ABCA12', 'NTPCR', 'PGPEP1', 'RNF19B', 'LCE1E', 'EPN3', 'SCN4B', 'ARVCF', 'FCHO2',
    'PANK2', 'TMEM8B', 'RRM2B', 'ANKRA2', 'ORAI3', 'PLCL2', 'SAC3D1', 'LIMK2', 'FBXO32', 'SCRIB',
    'BHLHE40', 'FCHSD2', 'PAQR7', 'TP53', 'MDM2', 'CCNG1', 'PRKAB1', 'PMAIP1', 'SYTL1', 'LRP1',
    'FHL2', 'SEMA3B', 'BMP7', 'FLRT2', 'PCBP4', 'TP53I11', 'SUSD6', 'CYFIP2', 'PTP4A1', 'PRDM1',
    'TNFRSF10A', 'MCC', 'HES2', 'SLC25A45', 'BORCS7', 'GBE1', 'PERP', 'TRAK1', 'GDF15', 'DRAM1',
    'SESN2', 'RAP2B', 'TNFRSF10D', 'NUPR1', 'KCNN4', 'SLC44A5', 'BTBD10', 'GPC1', 'PLLP', 'TRIP6',
    'BTG2', 'FBXO22', 'SLC30A1', 'RRAD', 'TSPAN11', 'PARD6G', 'KLHDC7A', 'SLC4A11', 'BTG3', 'HES1',
    'POU3F1', 'TSGA10', 'DDB2', 'ISCU', 'SPATA18', 'ZNF219', 'VWCE', 'PHPT1', 'LMNA', 'SLC9A1',
    'C17orf89', 'HRAS', 'PPFIBP1', 'UNC5B', 'GADD45A', 'PHLDA3', 'TGFA', 'ZNF337', 'DDIT4', 'PIDD1',
    'MLF2', 'STAT3', 'CAPN2', 'HSD17B3', 'PPM1J', 'UQCC1', 'PLK3', 'SERPINB5', 'TLR3', 'ACTA2',
    'RAD51C', 'PML', 'MR1', 'STK17A', 'CASP6', 'ICOSLG', 'PPP4R3A', 'VDR', 'TIGAR', 'SERTAD1',
    'TM7SF3', 'EDN2', 'SERPINE1', 'PTPRE', 'MYO6', 'STX6', 'CATSPERG', 'IGFBP7', 'PTAFR', 'YPEL3',
    'RPS27L', 'TRAF4', 'TMEM68', 'ALOX5', 'TNFAIP8', 'PVRL4', 'NEFL', 'TP73', 'CAV1', 'IL1B',
    'RALGDS', 'ZNF195', 'TNFRSF10B', 'TRIM22', 'WDR63', 'ARHGEF3', 'TSKU', 'RETSAT', 'NKAIN4',
    'TRIM32', 'CCNK', 'ISYNA1', 'RBM38', 'ZNF385A', 'TRIAP1', 'CES2', 'ZNF561', 'CERS5', 'PCNA',
    'REV3L', 'PCLO', 'TRIM38', 'CFLAR', 'JAG1', 'RGL1', 'ZNF488', 'ZMAT3', 'CMBL', 'ZNF79',
    'DDR1', 'ACYP2', 'RNASE7', 'PDE4C', 'TRIM5', 'CGB7', 'KRT8', 'RGS20', 'BAX', 'FBXW7',
    'ASCC3', 'DHRS3', 'APAF1', 'SFN', 'PGAP1', 'TYMSOS', 'CHST14', 'KSR1', 'RHOC', 'PGF',
    'HSPA4L', 'ACER2', 'DUSP14', 'APOBEC3H', 'TNFRSF10C', 'PLCXD2', 'AKAP9', 'COBLL1', 'LACC1',
    'RPS19', 'POLH', 'KITLG', 'ANXA4', 'E2F7', 'BCL2L1', 'TRIML2', 'PLEKHF1', 'CCDC51', 'CPEB2',
    'LPXN', 'SARS', 'PPM1D', 'SLC12A4', 'APOBEC3C', 'EPS8L2', 'BCL6', 'VCAN', 'PLTP', 'CDH8',
    'CPSF4', 'LRPAP1', 'SCIN', 'SULF2', 'ATF3', 'ASTN2', 'FAM210B', 'BLCAP', 'ADCK3', 'PLXNB1',
    'DUSP11', 'DNAJB2', 'MFAP3L', 'SCN3B', 'XPC', 'BBC3', 'CD82', 'GLS2', 'C17orf82', 'AK3',
    'PLXNB2', 'GCC2', 'DOCK8', 'MKNK2', 'SDC4', 'AEN', 'CCDC90B', 'CDIP1', 'GPX1', 'COL7A1',
    'ALDH1A3', 'PRKAB2', 'METTL8', 'DUSP5', 'MON2', 'SDPR', 'BLOC1S2', 'DYRK3', 'CPE', 'GRHL3',
    'CPEB4', 'BBS2', 'PRKX', 'PPP1R3C', 'DUSP7', 'MRPL49', 'SMAD3', 'FAS', 'EDA2R', 'CSF1',
    'HHAT', 'CSNK1G1', 'BTG1', 'PRODH', 'STEAP3', 'EBI3', 'MYBPHL', 'SNX2', 'GPR87', 'EPHA2',
    'DCP1B', 'IGDCC4', 'DGKA', 'CEL', 'PTPRU', 'ABHD4', 'EFNB1', 'MYLK', 'SOCS4', 'NINJ1',
    'FAM13C', 'ENC1', 'IKBIP', 'FAM49A', 'CLCA2', 'RGMA', 'ABTB2', 'EI24', 'MYOF', 'TAB3',
    'PLK2', 'FAM198B', 'FOSL1', 'LAPTM5', 'FAM84B', 'CLDN1', 'RGS16', 'ADGRG1', 'EML2', 'NFKBIA',
    'TCAIM', 'PSTPIP2', 'FAM212B', 'FUCA1', 'MAST4', 'GNAI1', 'CLP1', 'RND3', 'AIFM2', 'ENPP2',
    'NHLH2', 'TEP1', 'SESN1', 'FDXR', 'IER5', 'MICALL1', 'INPP1', 'CROT', 'RNF144B', 'AMOTL1',
    'ETV7', 'NLRP1', 'TET2', 'TP53I3', 'LIF', 'PADI4', 'NOTCH1', 'ITGA3', 'CYP4F3', 'S100A2',
    'AMZ2', 'FAM196A', 'NYNRIN', 'TEX9', 'TP53INP1', 'NADSYN1', 'PANK1', 'RABGGTA', 'KRT15',
    'DAPK1', 'SCN2A', 'ARC', 'FAM98C', 'P3H2', 'TMEM63B'
]

# Get the list of genes present in mRNA_data
genes_in_data = X_train.columns.to_list()

#Filter the TP53 target genes to include only the ones in the data
tp53_targets_final=list(set(tp53_targets).intersection(genes_in_data))

# Filter our datasets to keep only target genes
X_train = X_train[tp53_targets_final]
X_test = X_test[tp53_targets_final]


# Models

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV

In [21]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [22]:
# 2. Define the model
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# 3. Define the hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5],
    'reg_alpha': [0, 1]
}

# 4. Use cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# 5. Fit the model
grid_search.fit(X_train, y_train)

# 6. Evaluate the model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Best Parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'reg_alpha': 1, 'reg_lambda': 1, 'subsample': 0.8}
Accuracy: 0.8654708520179372
ROC AUC: 0.9267482517482518
              precision    recall  f1-score   support

           0       0.88      0.72      0.79        80
           1       0.86      0.94      0.90       143

    accuracy                           0.87       223
   macro avg       0.87      0.83      0.85       223
weighted avg       0.87      0.87      0.86       223



In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

# Define hyperparameter grid for tuning
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 0.5, 1.0],
    'bootstrap': [True]
}

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with cross-validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the best model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print("✅ Best Hyperparameters:", grid_search.best_params_)
print("🎯 Accuracy:", accuracy_score(y_test, y_pred))
print("📊 ROC AUC:", roc_auc_score(y_test, y_proba))
print("📋 Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
✅ Best Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
🎯 Accuracy: 0.8699551569506726
📊 ROC AUC: 0.9230769230769231
📋 Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.69      0.79        80
           1       0.85      0.97      0.91       143

    accuracy                           0.87       223
   macro avg       0.89      0.83      0.85       223
weighted avg       0.88      0.87      0.86       223



# Multiclass

In [73]:
y_train = y_train_multiclass
y_test = y_test_multiclass

In [74]:
print(f"Training data shape: {X_train.shape}")
print(f"Training labels distribution:\n{y_train.value_counts()}")
print()
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels distribution:\n{y_test.value_counts()}")

Training data shape: (891, 335)
Training labels distribution:
Variant_Type
SNP             498
WT              321
DEL              56
INS              14
DNP               2
Name: count, dtype: int64

Testing data shape: (223, 335)
Testing labels distribution:
Variant_Type
SNP             125
WT               80
DEL              14
INS               4
Name: count, dtype: int64


The number of samples with the 'DNP' mutation is very small, so we remove them to prevent their disproportionate influence on the classification model, which could negatively impact performance.

In [75]:
train_mask = y_train.iloc[:, 0] != 'DNP'
X_train = X_train.loc[train_mask].reset_index(drop=True)
y_train = y_train.loc[train_mask].reset_index(drop=True)

test_mask = y_test.iloc[:, 0] != 'DNP'
X_test = X_test.loc[test_mask].reset_index(drop=True)
y_test = y_test.loc[test_mask].reset_index(drop=True)

In [81]:
from sklearn.preprocessing import LabelEncoder

# Flatten and encode
y_train_flat = y_train.iloc[:, 0].values
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_flat)

y_test_flat = y_test.iloc[:, 0].values
y_test = label_encoder.transform(y_test_flat)

In [76]:
print(f"Training data shape: {X_train.shape}")
print(f"Training labels distribution:\n{y_train.value_counts()}")
print()
print(f"Testing data shape: {X_test.shape}")
print(f"Testing labels distribution:\n{y_test.value_counts()}")

Training data shape: (889, 335)
Training labels distribution:
Variant_Type
SNP             498
WT              321
DEL              56
INS              14
Name: count, dtype: int64

Testing data shape: (223, 335)
Testing labels distribution:
Variant_Type
SNP             125
WT               80
DEL              14
INS               4
Name: count, dtype: int64


In this multi-class classification task, the dataset is imbalanced, with some TP53 mutation types being much more common than others. To evaluate model performance fairly, we use average='weighted' for metrics like precision, recall, and F1 score.

This approach calculates the metric for each class and then averages them, weighted by the number of true instances per class. This ensures that more frequent classes have a greater impact on the final score, while still including performance on rare classes.

In [ ]:
num_classes = 4

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5],
    'reg_alpha': [0, 1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

xgb_model_multiclass = grid_search.best_estimator_
y_pred = xgb_model_multiclass.predict(X_test)
y_proba = xgb_model_multiclass.predict_proba(X_test)

print("Best Parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 288 candidates, totalling 1440 fits


In [ ]:
rf_model = RandomForestClassifier(random_state=42)

# Define hyperparameter grid for tuning
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 0.5, 1.0],
    'bootstrap': [True]
}

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with cross-validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train, y_train)

# Evaluate the best model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print("✅ Best Hyperparameters:", grid_search.best_params_)
print("🎯 Accuracy:", accuracy_score(y_test, y_pred))
print("📊 ROC AUC:", roc_auc_score(y_test, y_proba))
print("📋 Classification Report:\n", classification_report(y_test, y_pred))